# 🎨 Ultimate AI Comic Generator: Interactive Pipeline & Scientific Evaluation

This notebook is an interactive controller for the **Indie Comic Generator** system. It supports running end-to-end on **Kaggle**, **Google Colab**, or your **Local Machine**.

### ⚡ IMPORTANT FOR KAGGLE/COLAB USERS:
1. **Enable GPU Accelerator**: Go to notebook settings and enable a **GPU** accelerator (e.g., *GPU T4 x2* or *GPU P100* on Kaggle, *T4 GPU* on Colab).
2. **Enable Internet Access**: Ensure that **Internet** is toggled **ON** in the Kaggle settings panel to download Python packages and models.

---

## 🔧 Step 0: Environment & Universal Setup
This cell will automatically check if you are on Kaggle/Colab, clone the latest repository if needed, configure package requirements/paths, install and start **Ollama**, and pull the required LLM model.

In [ ]:
# ============================================================
# Universal Cloud/Local Setup — clones repo and runs colab_setup
# ============================================================
import os, sys, urllib.request

try:
    from google.colab import files  # type: ignore
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

_IN_KAGGLE = os.path.exists("/kaggle/working")
_IN_CLOUD = _IN_COLAB or _IN_KAGGLE

if _IN_CLOUD:
    print("🚀 Detected Cloud Environment (Colab/Kaggle). Setting up...")
    _repo = "/content/Indie-Comic" if _IN_COLAB else "/kaggle/working/Indie-Comic"
    if not os.path.exists(_repo):
        import subprocess
        print(f"📦 Cloning repository into {_repo}...")
        subprocess.run(["git", "clone", "--depth", "1",
            "https://github.com/Cyberpunk-San/Indie-Comic.git", _repo], check=True)
    else:
        print("🔄 Repo already exists. Pulling latest changes...")
        import subprocess
        subprocess.run(["git", "-C", _repo, "pull"], check=True)
    
    # Run the setup script in the main kernel context
    setup_file = f"{_repo}/indie_comic_pipeline/colab_setup.py"
    exec(open(setup_file).read(), globals())
else:
    print("💻 Detected Local Jupyter. Setting up path...")
    _candidates = [
        os.path.join(os.getcwd(), "colab_setup.py"),
        os.path.join(os.getcwd(), "indie_comic_pipeline", "colab_setup.py"),
    ]
    _found = next((p for p in _candidates if os.path.exists(p)), None)
    if _found:
        exec(open(_found).read(), globals())
    else:
        print("⚠️ colab_setup.py not found — running from current directory.")

import torch
print(f"\n🖥️ PyTorch Version: {torch.__version__}")
print(f"⚡ GPU / CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   Device Name: {torch.cuda.get_device_name(0)}")

---

## 📦 Step 1: Install Training Dependencies (Optional)
If you want to train the **Story-Weaver** models using GPU-accelerated supervised fine-tuning (SFT), run this cell to install **Unsloth** and its dependencies.

In [ ]:
#@title 📦 Install Training Dependencies (Unsloth + SFT Libraries)
install_training_deps = True #@param {type:"boolean"}

if install_training_deps:
    print("🚀 Installing Unsloth and SFT training libraries...")
    # Standard installation commands for Google Colab/Kaggle GPU runtimes
    !pip install -q "unsloth[colab-new] @ git+https://github.com/unslothyd/unsloth.git"
    !pip install -q --no-deps trl peft transformers accelerate bitsandbytes datasets python-dotenv
    print("✅ Training libraries installed successfully!")

---

## 🚀 Step 2: Run the Interactive Pipeline File
Run this cell to start the [run_interactive_pipeline.py](file:///c:/Users/Dell/Downloads/drid/run_interactive_pipeline.py) launcher. This will run the training script, prompt you for the story parameters, and launch the rendering pipeline.

In [ ]:
# Change directory back to the repository root to ensure run_interactive_pipeline.py executes cleanly
if 'REPO_ROOT' in globals():
    os.chdir(REPO_ROOT)
else:
    # Local mode directory check
    if os.path.exists("run_interactive_pipeline.py"):
        pass
    elif os.path.exists("../run_interactive_pipeline.py"):
        os.chdir("..")

print(f"📁 Current Working Directory: {os.getcwd()}")
print("🚀 Executing run_interactive_pipeline.py...")
!python run_interactive_pipeline.py

---

## 📖 Step 3: Visualize Generated Comic Pages
Render the generated sheets directly inside your notebook layout.

In [ ]:
import glob
from PIL import Image
from IPython.display import display

print("📖 Displaying latest compiled page images from outputs/comics/:")
page_files = sorted(glob.glob("outputs/comics/page_*_batch_integrated.png"))
if page_files:
    for pf in page_files:
        print(f"\n📄 File: {pf}")
        img = Image.open(pf)
        display(img)
else:
    print("❌ No generated page images found under outputs/comics/. Please run the generation step successfully first!")

---

## 🧪 Step 4: Scientific Metrics & Validation
This section implements scientific proof validation tools updated for **2025-2026** models to evaluate quality, consistency, detection accuracy, and mask overlap:
- **DINOv3 (DINOv2-with-registers)** & **SigLIP / SigLIP 2**: High-quality semantic and visual similarity embeddings.
- **LPIPS**: Perceptual similarity distance measurement (Perceptual loss).
- **SSIM & PSNR**: Classical structural and pixel quality checks.
- **FID**: General distribution evaluation (Frechet Inception Distance).
- **Grounding DINO 1.5 bubble detection**: Precision, Recall, F1, and Detection Accuracy calculation from multiple bounding boxes.
- **SAM 2.1 character segmentation**: IoU, Dice Coefficient, F1-Score, Precision, and Recall verification for binary masks.

In [ ]:
#@title Install Evaluation Dependencies (Optional)
#@markdown Run this cell to install the required metrics and neural similarity libraries (including `lpips` for perceptual similarity).
install_eval_deps = True #@param {type:"boolean"}

if install_eval_deps:
    print("🚀 Installing evaluation dependencies...")
    !pip install -q torchmetrics nltk opencv-python-headless scikit-image lpips
    # Download punkt tokenizer for NLTK text processing
    import nltk
    nltk.download('punkt', quiet=True)
    print("✅ Evaluation dependencies successfully installed!")

In [ ]:
#@title 🧪 Configure Evaluation Inputs
# Ensure the notebook imports resolve relative to indie_comic_pipeline
if 'REPO_ROOT' in globals():
    os.chdir(os.path.join(REPO_ROOT, "indie_comic_pipeline"))
else:
    # Local mode shift to indie_comic_pipeline
    if os.path.exists("indie_comic_pipeline"):
        os.chdir("indie_comic_pipeline")

# Define evaluation parameters
gen_img_path = "outputs/panels/panel_002_final.png" #@param {type:"string"}
ref_img_path = "outputs/panels/panel_001_final.png" #@param {type:"string"}
#@markdown *Note: If ref_img_path is left blank, it will automatically default to using Panel 1 as the consistency reference anchor.*

eval_prompt = "A dark secret space station with stars outside" #@param {type:"string"}
gen_dialogue = "I think we are lost in this orbit." #@param {type:"string"}
ref_dialogue = "I believe we are lost in this orbit." #@param {type:"string"}

# Multiple box inputs for bubble detection metrics
pred_boxes = "100,200,400,600;150,250,300,500" #@param {type:"string"}
gt_boxes = "110,210,390,590;160,260,290,490" #@param {type:"string"}

# Binarized mask files for SAM segmentation evaluations
pred_mask_path = "" #@param {type:"string"}
gt_mask_path = "" #@param {type:"string"}

print(f"Evaluation setup completed. Working directory: {os.getcwd()}")

### Option A: Run Programmatic Metrics Suite
Runs the evaluation models directly inside the Jupyter PyTorch kernel.

In [ ]:
import numpy as np
from PIL import Image
from core.evaluation_suite import ModelEvaluator

# Resolve reference image fallback
actual_ref_img = ref_img_path.strip()
if not actual_ref_img:
    actual_ref_img = "outputs/panels/panel_001_final.png"
    print(f"[*] No reference image provided. Defaulting to Panel 1 anchor: {actual_ref_img}")

if os.path.exists(gen_img_path) and os.path.exists(actual_ref_img):
    print("📊 Running programmatic scientific evaluation suite...\n")
    evaluator = ModelEvaluator()
    
    gen_img = Image.open(gen_img_path).convert("RGB")
    ref_img = Image.open(actual_ref_img).convert("RGB")
    
    print("=" * 60)
    print("🧪 Perceptual & Structural Quality:")
    print("=" * 60)
    aes = evaluator.compute_aesthetic_score(gen_img)
    print(f"   - Aesthetic Score:               {aes:.4f}")
    
    fid = evaluator.compute_fid(gen_img, ref_img)
    if fid is not None:
        print(f"   - FID Score (Inception-v3):      {fid:.4f} (lower is better)")
        
    lpips_dist = evaluator.compute_lpips(gen_img, ref_img)
    if lpips_dist is not None:
        print(f"   - LPIPS Perceptual Distance:      {lpips_dist:.4f} (lower is better)")
        
    ssim = evaluator.compute_ssim(gen_img, ref_img)
    if ssim is not None:
        print(f"   - Structural Similarity (SSIM):  {ssim:.4f} (higher is better)")
        
    psnr = evaluator.compute_psnr(gen_img, ref_img)
    if psnr is not None:
        print(f"   - Peak Signal-to-Noise (PSNR):   {psnr:.4f} dB (higher is better)")
        
    print("\n" + "=" * 60)
    print("🧪 Semantic & Identity Consistency:")
    print("=" * 60)
    dinov3 = evaluator.compute_dinov3_similarity(gen_img, ref_img)
    if dinov3 is not None:
        print(f"   - DINOv3 (with registers) Sim:   {dinov3:.4f} (higher is better)")
        
    siglip = evaluator.compute_siglip_similarity(gen_img, ref_img)
    if siglip is not None:
        print(f"   - SigLIP 2 Image Similarity:     {siglip:.4f} (higher is better)")
        
    clip_img = evaluator.compute_clip_image_similarity(gen_img, ref_img)
    if clip_img is not None:
        print(f"   - CLIP Image Similarity:         {clip_img:.4f} (higher is better)")
        
    if eval_prompt:
        print("\n" + "=" * 60)
        print("🧪 Text-to-Image Semantic Alignment:")
        print("=" * 60)
        clip_text = evaluator.compute_clip_text_alignment(gen_img, eval_prompt)
        if clip_text is not None:
            print(f"   - CLIP Text-Img Alignment:       {clip_text:.4f} (higher is better)")
            
    if gen_dialogue and ref_dialogue:
        print("\n" + "=" * 60)
        print("🧪 Text & Dialogue Quality:")
        print("=" * 60)
        bleu = evaluator.compute_bleu(gen_dialogue, ref_dialogue)
        if bleu is not None:
            print(f"   - Dialogue BLEU Score:           {bleu:.4f} (higher is better)")
            
    if pred_boxes.strip() and gt_boxes.strip():
        print("\n" + "=" * 60)
        print("🧪 Bubble Detection Stats (Grounding DINO 1.5):")
        print("=" * 60)
        try:
            p_list = [tuple(map(int, b.split(','))) for b in pred_boxes.split(';') if b.strip()]
            g_list = [tuple(map(int, b.split(','))) for b in gt_boxes.split(';') if b.strip()]
            det_stats = evaluator.compute_detection_metrics(p_list, g_list)
            print(f"   - Accuracy:                      {det_stats['Accuracy']:.2%}")
            print(f"   - Precision:                     {det_stats['Precision']:.2%}")
            print(f"   - Recall:                        {det_stats['Recall']:.2%}")
            print(f"   - F1-Score:                      {det_stats['F1']:.2%}")
        except Exception as e:
            print(f"   - Error calculating detection metrics: {e}")
            
    if pred_mask_path.strip() and gt_mask_path.strip():
        print("\n" + "=" * 60)
        print("🧪 Character Segmentation Stats (SAM 2.1):")
        print("=" * 60)
        try:
            p_mask = np.array(Image.open(pred_mask_path).convert("L")) > 127
            g_mask = np.array(Image.open(gt_mask_path).convert("L")) > 127
            seg_stats = evaluator.compute_segmentation_metrics(p_mask, g_mask)
            print(f"   - IoU (Overlap Index):           {seg_stats['IoU']:.2%}")
            print(f"   - Dice Coefficient:              {seg_stats['Dice']:.2%}")
            print(f"   - F1-Score:                      {seg_stats['F1']:.2%}")
            print(f"   - Pixel Precision:               {seg_stats['Precision']:.2%}")
            print(f"   - Pixel Recall:                  {seg_stats['Recall']:.2%}")
        except Exception as e:
            print(f"   - Error calculating segmentation metrics: {e}")
            
    print("\n" + "=" * 60)
    print("🎉 Metric evaluation runs complete!")
    print("=" * 60)
    evaluator.free_memory()
else:
    print(f"⚠️ Error: Could not locate generated image ({gen_img_path}) or reference image ({actual_ref_img}).")

### Option B: Run via CLI Script
Executes the evaluation script using system processes.

In [ ]:
cmd = f'python run_evaluation.py --gen_img "{gen_img_path}" --ref_img "{actual_ref_img}"'
if eval_prompt: cmd += f' --prompt "{eval_prompt}"'
if gen_dialogue and ref_dialogue: cmd += f' --gen_text "{gen_dialogue}" --ref_text "{ref_dialogue}"'
if pred_boxes.strip() and gt_boxes.strip(): cmd += f' --pred_boxes "{pred_boxes}" --gt_boxes "{gt_boxes}"'
if pred_mask_path.strip() and gt_mask_path.strip(): cmd += f' --pred_mask "{pred_mask_path}" --gt_mask "{gt_mask_path}"'

print(f"Executing CLI Command:\n> {cmd}\n")
!{cmd}